# Annual Report PDF -> RAG Ingestion Pipeline

This notebook turns the three annual report PDFs into a searchable, citable index:

1. **Azure Document Intelligence (`prebuilt-layout`)** extracts each PDF's text and, by
   requesting `output_content_format=markdown`, returns **tables already rendered as
   proper markdown tables** inline in the text — no manual table reconstruction needed.
2. Every detected **figure/image** is cropped out of the original page (via PyMuPDF),
   saved to disk under a **unique ID**, and sent to **Azure OpenAI (`gpt-5-mini`,
   multimodal)** to generate a factual text description.
3. That description is spliced back into the markdown **in place of the image**, tagged
   with the image's unique ID — so embeddings can "read" the image as text, and a
   chatbot can later look the ID back up to show the actual image alongside its answer.
4. Pages are chunked **page-wise, token-aware** (max 2000 tokens/chunk, 300-token
   overlap, never splitting a page's primary content) and embedded with
   **`text-embedding-3-large`** (3072 dimensions).
5. Chunks are indexed into **Azure AI Search**, carrying `source_pdf`, `page_start`/
   `page_end`, `image_ids`, and a `blob_url` for citations — ready for a chatbot to query.

Everything is cached to disk (raw Document Intelligence results, markdown, images,
metadata, per-page text, chunks+vectors) so re-running the notebook doesn't re-pay for
API calls you've already made.

Output layout (created under `output/`):

```
output/
  di_cache/<pdf>.json          raw Document Intelligence result (cache/resume)
  markdown/<pdf>.md            final markdown, tables inline, images replaced by [IMAGE ...] blocks
  images/<pdf>/<image_id>.png  cropped figure images
  metadata/<pdf>_images.jsonl  one JSON record per image: id, page, path, description
  pages/<pdf>_pages.jsonl      one JSON record per page: page_number, text (post figure-splice)
  chunks/<pdf>_chunks.jsonl    page-wise token-aware chunks, later including content_vector
```


## 0. Setup — config, paths, and Azure clients

In [1]:
import base64
import io
import json
import re
import time
import uuid
from pathlib import Path

from dotenv import load_dotenv
import os

load_dotenv()

# Hardcoded to this machine's project folder (change if you move the notebook).
PROJECT_ROOT = Path("/Users/princesinghal1/Desktop/Learning /RAG_Sehzan")

OUTPUT_DIR = PROJECT_ROOT / "output"
DI_CACHE_DIR = OUTPUT_DIR / "di_cache"
MARKDOWN_DIR = OUTPUT_DIR / "markdown"
IMAGES_DIR = OUTPUT_DIR / "images"
METADATA_DIR = OUTPUT_DIR / "metadata"
PAGES_DIR = OUTPUT_DIR / "pages"
CHUNKS_DIR = OUTPUT_DIR / "chunks"

for d in (DI_CACHE_DIR, MARKDOWN_DIR, IMAGES_DIR, METADATA_DIR, PAGES_DIR, CHUNKS_DIR):
    d.mkdir(parents=True, exist_ok=True)

AZURE_DI_ENDPOINT = os.environ["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]
AZURE_DI_KEY = os.environ["AZURE_DOCUMENT_INTELLIGENCE_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_VERSION = os.environ.get("AZURE_OPENAI_VERSION", "2025-01-01-preview")
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5-mini")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
EMBEDDING_DIMENSIONS = int(os.environ.get("EMBEDDING_DIMENSIONS", "3072"))

SEARCH_ENDPOINT = os.environ.get("SEARCH_ENDPOINT")
SEARCH_API_KEY = os.environ.get("SEARCH_API_KEY")
SEARCH_INDEX_NAME = os.environ.get("SEARCH_INDEX_NAME", "rag-assistant-bot")
BLOB_CONTAINER_URL = os.environ.get("BLOB_CONTAINER_URL", "").rstrip("/")

RENDER_DPI = 200       # resolution used when rasterizing PDF pages to crop out figures
FIGURE_PADDING_PX = 6  # extra pixels of padding kept around each detected figure's box

print("Project root:", PROJECT_ROOT)
print("Output dir:  ", OUTPUT_DIR)


Project root: /Users/princesinghal1/Desktop/Learning /RAG_Sehzan
Output dir:   /Users/princesinghal1/Desktop/Learning /RAG_Sehzan/output


In [2]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult, DocumentContentFormat
from openai import AzureOpenAI

di_client = DocumentIntelligenceClient(
    endpoint=AZURE_DI_ENDPOINT,
    credential=AzureKeyCredential(AZURE_DI_KEY),
)

aoai_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_VERSION,
)

print("Document Intelligence client ready ->", AZURE_DI_ENDPOINT)
print("Azure OpenAI client ready -> deployment:", AZURE_OPENAI_DEPLOYMENT)


Document Intelligence client ready -> https://document-intelligence-s0-new.cognitiveservices.azure.com/
Azure OpenAI client ready -> deployment: gpt-5-mini


In [3]:
def discover_pdfs():
    return sorted(PROJECT_ROOT.glob("*.pdf"))

PDF_PATHS = discover_pdfs()
for p in PDF_PATHS:
    print(f"{p.name:40s} {p.stat().st_size / 1e6:8.1f} MB")


Annual_Report_2021_22 1.pdf                  72.8 MB
Annual_Report_2022_23.pdf                    95.1 MB
Annual_Report_2023_24.pdf                    48.9 MB


## 1. Document Intelligence extraction (prebuilt-layout, markdown output)

`output_content_format=DocumentContentFormat.MARKDOWN` makes `result.content` a single
markdown string with tables already rendered as proper markdown tables. Figures are
returned separately in `result.figures`, each with the page + bounding polygon it lives
at and a `span` (offset/length) marking exactly where it sits inside `result.content` —
that's what lets us splice an image's description back into the right spot later.

These annual reports are 50-100+ MB. Rather than uploading the whole file in one request
(slow, and a single dropped connection throws away all the work), we **slice the PDF into
page-range batches with `pypdf`** and analyze one batch at a time. Each batch's raw result
is cached to `output/di_cache/<pdf>__p<start>-<end>.json` the moment it comes back, so an
interrupted run resumes from the last completed batch instead of re-paying for pages
that were already analyzed (pass `force=True` to redo everything).

**A note on the Document Intelligence pricing tier.** The **Free (F0)** tier caps every
request at the first 2 pages and 4 MB, no matter how many pages you send it — oversized
requests either get silently truncated to 2 pages or rejected outright
(`InvalidContentLength`). The **Standard (S0)** tier removes that cap (up to 2,000 pages /
500 MB per call). `analyze_pdf` below auto-detects both failure modes and **recursively
splits a batch in half** and retries until each piece succeeds — so this notebook works on
either tier — but on F0 that means roughly one Document Intelligence call *per 2 pages*,
which is slow and easily exceeds the free tier's 500-pages/month quota for ~250-page
annual reports. If your resource is on F0, upgrading it to S0 in the Azure portal
(Document Intelligence resource -> **Pricing tier**) is strongly recommended before a
full run — raise `BATCH_SIZE` below to 50-100 once you do, for far fewer, larger calls.


In [4]:
from pypdf import PdfReader, PdfWriter
from azure.core.exceptions import HttpResponseError

def get_page_count(pdf_path: Path) -> int:
    return len(PdfReader(str(pdf_path)).pages)


def slice_pdf_bytes(pdf_path: Path, start_page: int, end_page: int) -> bytes:
    '''Extract pages [start_page, end_page] (1-indexed, inclusive) into a standalone PDF
    in memory. Page content/mediabox is preserved exactly, so bounding polygons returned
    by Document Intelligence for this slice line up with the same page in the original file.'''
    reader = PdfReader(str(pdf_path))
    writer = PdfWriter()
    for i in range(start_page - 1, end_page):
        writer.add_page(reader.pages[i])
    buf = io.BytesIO()
    writer.write(buf)
    return buf.getvalue()


def analyze_pdf_batch(pdf_bytes: bytes) -> AnalyzeResult:
    poller = di_client.begin_analyze_document(
        "prebuilt-layout",
        body=pdf_bytes,
        content_type="application/pdf",
        output_content_format=DocumentContentFormat.MARKDOWN,
    )
    return poller.result()


def analyze_pdf_range(pdf_path: Path, start: int, end: int, stem: str,
                       force: bool = False) -> list[tuple[int, AnalyzeResult]]:
    '''Analyze pages [start, end] of pdf_path. If the service rejects the batch for being
    too large, or silently returns fewer pages than requested (both are normal on a
    constrained/free pricing tier), split the range in half and retry each half. Returns
    a list of (sub_batch_start_page, AnalyzeResult) covering the full [start, end] range.'''
    cache_path = DI_CACHE_DIR / f"{stem}__p{start}-{end}.json"
    want = end - start + 1

    if cache_path.exists() and not force:
        result = AnalyzeResult(json.loads(cache_path.read_text(encoding="utf-8")))
        if len(result.pages) == want:
            print(f"[{stem}] pages {start}-{end}: loading cached result")
            return [(start, result)]
        # stale/incomplete cache (e.g. tier limits changed) -> fall through and re-split

    if start == end:
        print(f"[{stem}] page {start}: calling Document Intelligence...")
    else:
        print(f"[{stem}] pages {start}-{end}: calling Document Intelligence...")

    try:
        pdf_bytes = slice_pdf_bytes(pdf_path, start, end)
        result = analyze_pdf_batch(pdf_bytes)
    except HttpResponseError as e:
        if start == end:
            raise RuntimeError(
                f"Page {start} of {stem} could not be analyzed even alone "
                f"(likely needs the Standard S0 pricing tier): {e}"
            ) from e
        mid = (start + end) // 2
        print(f"  [warn] batch too large ({e.error.code if e.error else e}), splitting in half")
        return (analyze_pdf_range(pdf_path, start, mid, stem, force)
                + analyze_pdf_range(pdf_path, mid + 1, end, stem, force))

    if len(result.pages) < want:
        if start == end:
            raise RuntimeError(f"Document Intelligence returned no result for page {start} of {stem}.")
        mid = (start + end) // 2
        print(f"  [warn] only {len(result.pages)}/{want} pages returned "
              f"(likely the Free F0 tier's 2-page cap), splitting and retrying")
        return (analyze_pdf_range(pdf_path, start, mid, stem, force)
                + analyze_pdf_range(pdf_path, mid + 1, end, stem, force))

    cache_path.write_text(json.dumps(result.as_dict()), encoding="utf-8")
    print(f"  -> {len(result.pages)} pages, {len(result.figures or [])} figures")
    return [(start, result)]


def analyze_pdf(pdf_path: Path, batch_size: int = 20, max_pages: int | None = None,
                 force: bool = False) -> list[tuple[int, AnalyzeResult]]:
    '''Returns a list of (batch_start_page, AnalyzeResult) covering the whole document (or
    the first max_pages, when set), in order. batch_start_page is the 1-indexed page number
    (in the ORIGINAL pdf) that result.pages[0] corresponds to -- used later to translate
    each figure's local page number back to its true page number.'''
    stem = pdf_path.stem
    total_pages = get_page_count(pdf_path)
    if max_pages:
        total_pages = min(total_pages, max_pages)

    batches = []
    for start in range(1, total_pages + 1, batch_size):
        end = min(start + batch_size - 1, total_pages)
        batches.extend(analyze_pdf_range(pdf_path, start, end, stem, force=force))
    return batches


## 2. Cropping figures out of the original page (PyMuPDF)

Document Intelligence tells us *where* a figure is (page number + polygon, in inches),
but not the pixels themselves. We rasterize the source page with PyMuPDF at `RENDER_DPI`
and crop out the figure's bounding box, converting inches -> pixels using that same DPI.
Each page is only rendered once and cached in memory even if it has multiple figures.


In [5]:
import fitz  # PyMuPDF
from PIL import Image

_page_image_cache: dict[tuple[str, int], "Image.Image"] = {}

def get_rendered_page(pdf_path: Path, page_number: int) -> "Image.Image":
    '''Rasterize one PDF page (1-indexed) at RENDER_DPI, cached per (pdf, page) for this run.'''
    key = (str(pdf_path), page_number)
    if key in _page_image_cache:
        return _page_image_cache[key]
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number - 1)
    zoom = RENDER_DPI / 72
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    doc.close()
    _page_image_cache[key] = img
    return img


def polygon_to_pixel_bbox(polygon, unit) -> tuple[float, float, float, float]:
    '''Convert a flat [x1,y1,x2,y2,...] DI polygon to a pixel box at RENDER_DPI.
    Document Intelligence reports PDF coordinates in inches; pixel = inch * RENDER_DPI.'''
    xs, ys = polygon[0::2], polygon[1::2]
    scale = RENDER_DPI if unit == "inch" else RENDER_DPI / 72
    return min(xs) * scale, min(ys) * scale, max(xs) * scale, max(ys) * scale


def crop_figure_image(pdf_path: Path, page_number: int, polygon, unit,
                       padding: int = FIGURE_PADDING_PX) -> "Image.Image":
    page_img = get_rendered_page(pdf_path, page_number)
    x0, y0, x1, y1 = polygon_to_pixel_bbox(polygon, unit)
    x0 = max(0, x0 - padding)
    y0 = max(0, y0 - padding)
    x1 = min(page_img.width, x1 + padding)
    y1 = min(page_img.height, y1 + padding)
    return page_img.crop((int(x0), int(y0), int(x1), int(y1)))


## 3. Describing each figure with GPT-5-mini (Azure OpenAI, multimodal)

The cropped figure is base64-encoded and sent as an image to the chat completions API.
We also pass Document Intelligence's own caption for that figure (when it detected one)
as extra context.

`gpt-5-mini` is a *reasoning* model: its `max_completion_tokens` budget is shared between
hidden reasoning tokens and the actual visible answer. Left alone, it sometimes spends the
**entire** budget reasoning and returns empty text. Passing `reasoning_effort="low"` fixes
this (verified: with it, `reasoning_tokens` drops to 0 and the real caption comes through;
without it, some images silently returned nothing). We still treat an empty response as a
failure worth retrying, and fall back gracefully if a parameter isn't accepted by the
deployment.


In [6]:
def _chat_with_image(prompt: str, image_b64: str, max_tokens: int = 700):
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        ],
    }]
    kwargs = {
        "model": AZURE_OPENAI_DEPLOYMENT,
        "messages": messages,
        "max_completion_tokens": max_tokens,
        "reasoning_effort": "low",
    }
    while True:
        try:
            return aoai_client.chat.completions.create(**kwargs)
        except Exception as e:
            msg = str(e)
            if "reasoning_effort" in kwargs and "reasoning_effort" in msg:
                del kwargs["reasoning_effort"]
            elif "max_completion_tokens" in kwargs and (
                "max_completion_tokens" in msg or "Unsupported parameter" in msg
            ):
                kwargs["max_tokens"] = kwargs.pop("max_completion_tokens")
            else:
                raise


def describe_image_with_gpt(image: "Image.Image", context_hint: str | None = None,
                             max_retries: int = 3) -> str:
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    image_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    prompt = (
        "You are extracting information from a page of a corporate annual report for a "
        "retrieval system. Describe this image factually so someone who cannot see it can "
        "understand what it shows.\n"
        "- If it is a chart, graph, or infographic: name the chart type and transcribe the "
        "key labels, categories, numbers, and trend exactly as shown.\n"
        "- If it is a photo, logo, diagram, or illustration: describe the visible content "
        "and any visible text.\n"
        "- If it is a portrait/photo of a person and the nearby document text below names "
        "them (and/or gives a title/role), lead with that name and title, then describe "
        "the photo. Do not guess a name or identity that isn't given in the text.\n"
        "Keep it factual and under 6 sentences. Do not speculate about anything not visible."
    )
    if context_hint:
        prompt += f"\n\nNearby document text for context: {context_hint.strip()[:800]}"

    last_err = None
    for attempt in range(max_retries):
        try:
            resp = _chat_with_image(prompt, image_b64)
            text = (resp.choices[0].message.content or "").strip()
            if not text:
                raise ValueError(
                    f"empty response (finish_reason={resp.choices[0].finish_reason}); "
                    f"the reasoning budget likely consumed all of max_completion_tokens"
                )
            return text
        except Exception as e:
            last_err = e
            print(f"  [warn] GPT captioning attempt {attempt + 1} failed: {e}")
            time.sleep(2 * (attempt + 1))
    return f"[Image description unavailable after {max_retries} attempts: {last_err}]"


## 4. Splicing figure descriptions back into the markdown

Each figure has one or more `spans` (character offset + length into `result.content`)
marking where it sits in the text. We replace that exact span with a tagged block:

```
[IMAGE id=<unique_id> page=<n> file=<path to saved png>]
<GPT-generated description>
[/IMAGE]
```

Replacements are applied **back-to-front** (highest offset first) so earlier offsets
in the string stay valid while later ones are being replaced. The image itself is saved
to `output/images/<pdf>/<unique_id>.png`, and a matching record (id, page, path,
description) is appended to `output/metadata/<pdf>_images.jsonl` — that JSONL is the
lookup table a chatbot will later use to attach the real image to an answer.

Because we analyze the PDF in page-range batches (see step 1), a figure's
`region.page_number` is local to its batch (starting at 1). `page_offset` (the batch's
true starting page minus 1) translates it back to the real page number in the original
file, which is what we use both to crop from the original PDF and to record in metadata.

**Naming people in photos.** GPT can't recognize who's in a photo from pixels alone — but
annual reports usually print the person's name right next to their portrait (e.g. a
`<figcaption>Dr Urvish Shah</figcaption>` block). Document Intelligence doesn't always
link that caption to the figure via `fig.caption`, so in addition to it, we pull the raw
text immediately **before and after** the figure's own span in `result.content` (a small
character window) and pass that along as extra context. That's usually enough for GPT to
pick up the adjacent name/title/caption and fold it into the description instead of
writing a purely generic "portrait of a man in a suit."


In [7]:
IMAGE_BLOCK_TEMPLATE = (
    "\n\n[IMAGE id={image_id} page={page_number} file={rel_path}]\n"
    "{description}\n"
    "[/IMAGE]\n\n"
)

_TAG_RE = re.compile(r"<!--.*?-->|</?figure>|</?figcaption>", re.DOTALL)

def get_nearby_text(content: str, fig: "DocumentFigure", window: int = 300) -> str | None:
    '''Grab the raw text immediately surrounding a figure's span (e.g. an adjacent name/
    caption/title that Document Intelligence didn't explicitly link via fig.caption).'''
    if not fig.spans:
        return None
    start = min(s.offset for s in fig.spans)
    end = max(s.offset + s.length for s in fig.spans)
    before = content[max(0, start - window):start]
    after = content[end:end + window]
    text = _TAG_RE.sub(" ", f"{before} {after}")
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def process_figures(result: AnalyzeResult, pdf_path: Path,
                     page_offset: int = 0) -> tuple[str, list[dict], list[tuple[int, str]]]:
    '''Returns (full_batch_markdown, image_records, page_texts). page_texts is a list of
    (true_page_number, page_markdown) -- the same figure-replacement splicing applied to
    full_batch_markdown, but sliced per page using each DocumentPage's own span -- so later
    steps (chunking) can build page-accurate, page-bounded chunks without re-parsing markup.'''
    stem = pdf_path.stem
    original_content = result.content
    figures = result.figures or []
    pages_by_number = {p.page_number: p for p in result.pages}

    img_dir = IMAGES_DIR / stem
    img_dir.mkdir(parents=True, exist_ok=True)

    replacements = []  # (offset, length, replacement_text)
    image_records = []

    for fig in figures:
        if not fig.bounding_regions:
            continue
        region = fig.bounding_regions[0]
        page = pages_by_number.get(region.page_number)
        if page is None:
            continue
        true_page_number = region.page_number + page_offset

        try:
            cropped = crop_figure_image(pdf_path, true_page_number, region.polygon, page.unit)
        except Exception as e:
            print(f"  [warn] could not crop figure {fig.id} on page {true_page_number}: {e}")
            continue

        image_id = uuid.uuid4().hex[:12]
        image_path = img_dir / f"{image_id}.png"
        cropped.save(image_path, "PNG")

        caption_hint = fig.caption.content if fig.caption else None
        nearby_hint = get_nearby_text(original_content, fig)
        context_hint = " | ".join(h for h in (caption_hint, nearby_hint) if h) or None
        description = describe_image_with_gpt(cropped, context_hint=context_hint)

        rel_path = str(image_path.relative_to(PROJECT_ROOT))
        image_records.append({
            "image_id": image_id,
            "source_pdf": pdf_path.name,
            "page_number": true_page_number,
            "image_path": rel_path,
            "description": description,
        })

        block = IMAGE_BLOCK_TEMPLATE.format(
            image_id=image_id, page_number=true_page_number,
            rel_path=rel_path, description=description,
        )
        for span in fig.spans:
            replacements.append((span.offset, span.length, block))

        print(f"  [{stem}] figure {fig.id} -> image {image_id} (page {true_page_number}) captioned")

    full_content = original_content
    for offset, length, block in sorted(replacements, key=lambda r: r[0], reverse=True):
        full_content = full_content[:offset] + block + full_content[offset + length:]

    # Per-page slices: cut the ORIGINAL content at each page's own span, then splice in
    # just the replacements that land inside that page's range (offsets re-based to the slice).
    page_texts = []
    for page in result.pages:
        if not page.spans:
            continue
        p_start = min(s.offset for s in page.spans)
        p_end = max(s.offset + s.length for s in page.spans)
        page_slice = original_content[p_start:p_end]
        page_repls = [(o - p_start, l, b) for o, l, b in replacements if p_start <= o < p_end]
        for offset, length, block in sorted(page_repls, key=lambda r: r[0], reverse=True):
            page_slice = page_slice[:offset] + block + page_slice[offset + length:]
        page_texts.append((page.page_number + page_offset, page_slice))

    return full_content, image_records, page_texts


## 5. Orchestration — run the full pipeline per PDF

`process_pdf` ties it all together and is **idempotent**: if a PDF's markdown + image
metadata already exist on disk, it skips re-processing (pass `force=True` to redo it).

Set `TEST_MODE = True` first and run on a handful of pages to validate the pipeline
end-to-end (and see what it costs/takes) before flipping it to `False` for a full run.


In [8]:
def process_pdf(pdf_path: Path, batch_size: int = 20, max_pages: int | None = None,
                force: bool = False) -> None:
    stem = pdf_path.stem
    md_path = MARKDOWN_DIR / f"{stem}.md"
    meta_path = METADATA_DIR / f"{stem}_images.jsonl"
    pages_path = PAGES_DIR / f"{stem}_pages.jsonl"

    if md_path.exists() and meta_path.exists() and pages_path.exists() and not force:
        print(f"[{stem}] already processed -> skipping (pass force=True to redo)")
        return

    batches = analyze_pdf(pdf_path, batch_size=batch_size, max_pages=max_pages, force=force)

    markdown_parts, all_image_records, all_page_texts = [], [], []
    for batch_start_page, result in batches:
        md_part, image_records, page_texts = process_figures(
            result, pdf_path, page_offset=batch_start_page - 1
        )
        markdown_parts.append(md_part)
        all_image_records.extend(image_records)
        all_page_texts.extend(page_texts)

    final_markdown = "\n\n".join(markdown_parts)
    md_path.write_text(final_markdown, encoding="utf-8")
    with open(meta_path, "w", encoding="utf-8") as f:
        for rec in all_image_records:
            f.write(json.dumps(rec) + "\n")
    with open(pages_path, "w", encoding="utf-8") as f:
        for page_number, text in sorted(all_page_texts, key=lambda p: p[0]):
            f.write(json.dumps({"page_number": page_number, "text": text}) + "\n")

    print(f"[{stem}] saved -> {md_path.name} ({len(final_markdown):,} chars), "
          f"{len(all_image_records)} images -> {meta_path.name}, "
          f"{len(all_page_texts)} pages -> {pages_path.name}")


In [9]:
TEST_MODE = False    # True = only analyze a few pages per PDF (cheap smoke test)
TEST_MAX_PAGES = None  # page cap used when TEST_MODE is True
BATCH_SIZE = 20      # pages per Document Intelligence call (keeps each upload small)

for pdf_path in PDF_PATHS:
    process_pdf(pdf_path, batch_size=BATCH_SIZE, max_pages=TEST_MAX_PAGES if TEST_MODE else None)


[Annual_Report_2021_22 1] pages 1-20: calling Document Intelligence...
  -> 20 pages, 88 figures
[Annual_Report_2021_22 1] pages 21-40: calling Document Intelligence...
  -> 20 pages, 33 figures
[Annual_Report_2021_22 1] pages 41-60: calling Document Intelligence...
  -> 20 pages, 42 figures
[Annual_Report_2021_22 1] pages 61-80: calling Document Intelligence...
  -> 20 pages, 34 figures
[Annual_Report_2021_22 1] pages 81-100: calling Document Intelligence...
  -> 20 pages, 45 figures
[Annual_Report_2021_22 1] pages 101-120: calling Document Intelligence...
  -> 20 pages, 32 figures
[Annual_Report_2021_22 1] pages 121-140: calling Document Intelligence...
  -> 20 pages, 40 figures
[Annual_Report_2021_22 1] pages 141-160: calling Document Intelligence...
  -> 20 pages, 42 figures
[Annual_Report_2021_22 1] pages 161-180: calling Document Intelligence...
  -> 20 pages, 10 figures
[Annual_Report_2021_22 1] pages 181-200: calling Document Intelligence...
  -> 20 pages, 1 figures
[Annual_Rep

KeyboardInterrupt: 

Once the smoke test above looks right (check `output/markdown/*.md` and
`output/images/*/*.png`), flip `TEST_MODE = False` above and re-run that cell for the
full documents. Because `analyze_pdf` and `process_pdf` both cache to disk, re-running
after a full pass won't repeat work or cost unless you pass `force=True`.


## 6. Page-wise, token-aware chunking for retrieval

This reads `output/pages/<pdf>_pages.jsonl` (written by `process_pdf` in step 5) — one
record per page, in original page order — and builds chunks for embedding:

- Chunks are built **page by page**: whole pages are appended to the current chunk until
  adding the next page would push it past **2000 tokens** (counted with `tiktoken`'s
  `cl100k_base` encoding, the same one `text-embedding-3-large` uses). A page is never cut
  in half for a chunk's primary content.
- Each new chunk is seeded with the **last ~300 tokens** of the previous chunk's text as
  overlap, so context carries across a chunk boundary — this is the one place a page's
  text can be split, since overlap is deliberately-duplicated trailing context, not a
  page's primary home.
- Every chunk records `source_pdf`, `page_start`/`page_end` (the page range it covers),
  any `image_ids` found inside it (from the `[IMAGE id=...]` blocks step 4 spliced in),
  and a `blob_url` built from `BLOB_CONTAINER_URL` + the pdf filename + page anchor — fill
  in `BLOB_CONTAINER_URL` in `.env` once the PDFs are uploaded to blob storage, so the
  chatbot UI can cite "page 12 of Annual_Report_2023_24.pdf" with a working link.
- A single page that alone exceeds 2000 tokens is kept whole anyway (a chunk can exceed
  the budget in that one case) rather than ever splitting a page's primary content.


In [ ]:
import tiktoken

IMAGE_ID_RE = re.compile(r"\[IMAGE id=(\S+) page=")
_ENCODING = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(_ENCODING.encode(text))


def build_blob_url(source_pdf: str, page_number: int) -> str | None:
    if not BLOB_CONTAINER_URL:
        return None
    return f"{BLOB_CONTAINER_URL}/{source_pdf}#page={page_number}"


def load_pages(pdf_stem: str) -> list[tuple[int, str]]:
    pages_path = PAGES_DIR / f"{pdf_stem}_pages.jsonl"
    pages = []
    with open(pages_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            pages.append((rec["page_number"], rec["text"]))
    return sorted(pages, key=lambda p: p[0])


def chunk_pages(pages: list[tuple[int, str]], source_pdf: str,
                max_tokens: int = 2000, overlap_tokens: int = 300) -> list[dict]:
    '''Greedily pack whole pages into a chunk up to max_tokens, never splitting a page's
    primary content; each new chunk is seeded with the last overlap_tokens of the
    previous chunk's text for continuity across the boundary.'''
    chunks: list[dict] = []
    current_pages: list[tuple[int, str]] = []
    current_tokens = 0
    carry_text = ""

    def flush():
        nonlocal current_pages, current_tokens, carry_text
        if not current_pages:
            return
        page_numbers = [p for p, _ in current_pages]
        body = "\n\n".join(t for _, t in current_pages)
        full_text = f"{carry_text}\n\n{body}".strip() if carry_text else body.strip()
        page_start, page_end = min(page_numbers), max(page_numbers)
        chunks.append({
            "chunk_id": uuid.uuid4().hex[:12],
            "source_pdf": source_pdf,
            "page_start": page_start,
            "page_end": page_end,
            "text": full_text,
            "image_ids": IMAGE_ID_RE.findall(full_text),
            "blob_url": build_blob_url(source_pdf, page_start),
        })
        tokens = _ENCODING.encode(full_text)
        tail = tokens[-overlap_tokens:] if len(tokens) > overlap_tokens else tokens
        carry_text = _ENCODING.decode(tail)
        current_pages = []
        current_tokens = 0

    carry_tokens = 0
    for page_number, text in pages:
        page_tokens = count_tokens(text)
        if page_tokens > max_tokens:
            print(f"  [warn] page {page_number} alone is {page_tokens} tokens (> {max_tokens}); "
                  f"keeping it whole in its own chunk rather than splitting it")
        if current_pages and (current_tokens + carry_tokens + page_tokens > max_tokens):
            flush()
            carry_tokens = count_tokens(carry_text)
        current_pages.append((page_number, text))
        current_tokens += page_tokens
    flush()
    return chunks


all_chunks_by_pdf: dict[str, list[dict]] = {}
for pages_path in PAGES_DIR.glob("*_pages.jsonl"):
    stem = pages_path.stem.removesuffix("_pages")
    source_pdf = stem + ".pdf"
    pages = load_pages(stem)
    chunks = chunk_pages(pages, source_pdf=source_pdf)
    all_chunks_by_pdf[stem] = chunks

    out_path = CHUNKS_DIR / f"{stem}_chunks.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c) + "\n")
    print(f"{stem}: {len(pages)} pages -> {len(chunks)} chunks -> {out_path.name}")


## 7. Answer-time image lookup

When a chatbot retrieves a chunk and finds an `[IMAGE id=...]` reference inside it, it
needs to resolve that ID back to the actual saved image file to show the user. This
loads every `*_images.jsonl` metadata file into one in-memory index for that lookup.


In [ ]:
def load_all_image_metadata() -> dict[str, dict]:
    records = {}
    for meta_path in METADATA_DIR.glob("*_images.jsonl"):
        with open(meta_path, encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                records[rec["image_id"]] = rec
    return records

IMAGE_INDEX = load_all_image_metadata()
print(f"Loaded {len(IMAGE_INDEX)} image records")

def get_image_for_answer(image_id: str) -> dict | None:
    '''Resolve an [IMAGE id=...] reference found in a retrieved chunk back to its
    saved file + description, so the chatbot can attach the real image to its answer.'''
    return IMAGE_INDEX.get(image_id)

list(IMAGE_INDEX.items())[:2]


## 8. Embedding each chunk (`text-embedding-3-large`, 3072 dimensions)

Each chunk's `text` is sent to the Azure OpenAI embeddings endpoint. Calls are batched
(several chunks per request) for efficiency, with retry/backoff for transient errors.
The resulting vector is attached to the chunk record as `content_vector` and the whole
set is re-saved to `output/chunks/<pdf>_chunks.jsonl` (now including the vectors).

> **Note:** as of this writing, the `text-embedding-3-large` deployment on this Azure
> OpenAI resource isn't live yet (`DeploymentNotFound`). This cell will raise clearly if
> that's still the case — finish deploying it in Azure AI Foundry, confirm
> `AZURE_OPENAI_EMBEDDING_DEPLOYMENT` in `.env` matches its exact name, then re-run.


In [ ]:
def embed_texts(texts: list[str], batch_size: int = 16, max_retries: int = 3) -> list[list[float]]:
    all_vectors: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        last_err = None
        for attempt in range(max_retries):
            try:
                resp = aoai_client.embeddings.create(
                    model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    input=batch,
                    dimensions=EMBEDDING_DIMENSIONS,
                )
                all_vectors.extend(d.embedding for d in resp.data)
                break
            except Exception as e:
                last_err = e
                print(f"  [warn] embedding batch {i // batch_size} attempt {attempt + 1} failed: {e}")
                time.sleep(2 * (attempt + 1))
        else:
            raise RuntimeError(f"embedding batch {i // batch_size} failed after {max_retries} attempts") from last_err
    return all_vectors


def embed_chunks_for_pdf(stem: str) -> list[dict]:
    chunks_path = CHUNKS_DIR / f"{stem}_chunks.jsonl"
    chunks = [json.loads(line) for line in chunks_path.read_text(encoding="utf-8").splitlines()]
    vectors = embed_texts([c["text"] for c in chunks])
    for c, v in zip(chunks, vectors):
        c["content_vector"] = v
    with open(chunks_path, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c) + "\n")
    print(f"[{stem}] embedded {len(chunks)} chunks -> {chunks_path.name}")
    return chunks


all_embedded_chunks: list[dict] = []
for pages_path in PAGES_DIR.glob("*_pages.jsonl"):
    stem = pages_path.stem.removesuffix("_pages")
    all_embedded_chunks.extend(embed_chunks_for_pdf(stem))

print(f"Total embedded chunks across all PDFs: {len(all_embedded_chunks)}")


## 9. Indexing into Azure AI Search

Creates (or recreates) a vector-search-enabled index and uploads every embedded chunk
as a search document. Fields per document:

| Field | Type | Purpose |
|---|---|---|
| `id` | key (string) | unique per chunk |
| `source_pdf` | string, filterable | which report this came from |
| `page_start` / `page_end` | int32, filterable/sortable | page range for citations |
| `text` | searchable string | the chunk's text (keyword + semantic search) |
| `image_ids` | collection(string) | figure ids inside this chunk, for `get_image_for_answer` |
| `blob_url` | string | citation link into blob storage (page-anchored) |
| `content_vector` | collection(single), 3072-dim | the embedding, for vector search |

`recreate_search_index()` **deletes** the index first if it already exists (matching how
this notebook's index was reset earlier) so re-running the pipeline always starts from a
clean, consistent schema — pass `recreate=False` to just upload into an existing index.


In [ ]:
from azure.core.credentials import AzureKeyCredential as _AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, VectorSearchProfile, HnswAlgorithmConfiguration,
)

search_index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT, credential=_AzureKeyCredential(SEARCH_API_KEY)
)


def build_search_index_schema() -> SearchIndex:
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SimpleField(name="source_pdf", type=SearchFieldDataType.String, filterable=True),
        SimpleField(name="page_start", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
        SimpleField(name="page_end", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
        SearchableField(name="text", type=SearchFieldDataType.String),
        SimpleField(name="image_ids", type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                    filterable=True),
        SearchField(
            name="content_vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=EMBEDDING_DIMENSIONS,
            vector_search_profile_name="default-vector-profile",
        ),
    ]
    vector_search = VectorSearch(
        profiles=[VectorSearchProfile(name="default-vector-profile",
                                       algorithm_configuration_name="default-hnsw")],
        algorithms=[HnswAlgorithmConfiguration(name="default-hnsw")],
    )
    return SearchIndex(name=SEARCH_INDEX_NAME, fields=fields, vector_search=vector_search)


def recreate_search_index(recreate: bool = True) -> None:
    if recreate and SEARCH_INDEX_NAME in search_index_client.list_index_names():
        print(f"Deleting existing index '{SEARCH_INDEX_NAME}'...")
        search_index_client.delete_index(SEARCH_INDEX_NAME)
    search_index_client.create_index(build_search_index_schema())
    print(f"Index '{SEARCH_INDEX_NAME}' is ready.")


def upload_chunks_to_search(chunks: list[dict], batch_size: int = 100) -> None:
    search_client = SearchClient(
        endpoint=SEARCH_ENDPOINT, index_name=SEARCH_INDEX_NAME,
        credential=_AzureKeyCredential(SEARCH_API_KEY),
    )
    docs = [{
        "id": c["chunk_id"],
        "source_pdf": c["source_pdf"],
        "page_start": c["page_start"],
        "page_end": c["page_end"],
        "text": c["text"],
        "image_ids": c.get("image_ids", []),
        "content_vector": c["content_vector"],
    } for c in chunks]

    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        result = search_client.upload_documents(documents=batch)
        failed = [r for r in result if not r.succeeded]
        if failed:
            print(f"  [warn] {len(failed)}/{len(batch)} documents failed in batch {i // batch_size}")
        else:
            print(f"  uploaded batch {i // batch_size} ({len(batch)} docs)")


recreate_search_index(recreate=True)
upload_chunks_to_search(all_embedded_chunks)
print(f"Indexed {len(all_embedded_chunks)} chunks into '{SEARCH_INDEX_NAME}'.")


## Next steps (not in this notebook)

- **Build the chatbot's retrieval step**: embed the user's question the same way (step 8),
  run a vector (or hybrid vector+keyword) search against `SEARCH_INDEX_NAME`, and pass the
  top chunks' `text` to the answering model as context.
- **Citations in the UI**: each search hit carries `source_pdf`, `page_start`/`page_end`,
  and `blob_url` — enough to render "Annual_Report_2023_24.pdf, p. 42" as a clickable
  citation once the PDFs are uploaded to blob storage and `BLOB_CONTAINER_URL` is set.
- **Inline images**: scan a matched chunk's `image_ids` and call `get_image_for_answer`
  (step 7) to attach the real cropped image(s) next to the answer.
- Re-run `process_pdf(..., force=True)` only for PDFs where you change the extraction
  logic, and `recreate_search_index(recreate=False)` if you just want to add more chunks
  without wiping the index.
